# Simulate

Declare a run, send it, save it. A run is five names and a few numbers (`RunSpec`); `resolve` loads what the names point at and lays out the calls, `execute` runs the batch, reruns skipped calls a capped number of times and saves twice (archived, then plain).

**Cost:** every call is one API request. `SPECS["fake"]` runs offline; `sample=200` variants land under a `_subset=200` stem and never overwrite a full run.

In [ ]:
from dotenv import load_dotenv

from panel import paths
from panel.analysis import validate
from panel.run import ResultStore, RunSpec, execute, resolve
from panel.run.registry import client_class

load_dotenv()

In [ ]:
# Run setup - the only place a run is declared.

BASE = RunSpec(
    dataset="sod", description="self_v1", topic="election_2021", framing="cs",
    client="OpenRouter", model="gpt-4.1-nano", shape="VProb", temperature=0.0,
)

SPECS = {
    "nano":           BASE,
    "nano-trial":     BASE.variant(sample=200),
    "4o-mini":        BASE.variant(model="gpt-4o-mini"),
    "4.1":            BASE.variant(model="gpt-4.1"),
    "4o":             BASE.variant(model="gpt-4o"),
    "gemini":         BASE.variant(model="gemini-3-flash-preview"),
    "sonnet":         BASE.variant(model="claude-sonnet-4-5"),
    "4o-openai":      BASE.variant(client="OpenAI", model="gpt-4o"),
    "4o-pick":        BASE.variant(client="OpenAI", model="gpt-4o", shape="VPick"),
    "v2":             BASE.variant(description="self_v2"),
    "grid":           BASE.variant(dataset="grid"),
    "latent-citizen": BASE.variant(dataset="regions", description="latent_citizen_v1", client="OpenAI", repeats=10),
    "latent-direct":  BASE.variant(dataset="regions", description="latent_direct_v1", client="OpenAI", repeats=10),
    "fake":           BASE.variant(client="Fake", model="fake-model", sample=20),
}

SPEC = SPECS["fake"]

In [ ]:
RUN = resolve(SPEC)
STORE = ResultStore(paths.RESULTS_DIR, SPEC.topic)
CLIENT = client_class(SPEC.client)()

print(f"{RUN.stem}: {len(RUN.calls)} calls over {len(RUN.rows)} subjects -> {STORE.paths(RUN.stem).results_csv}")

## Prompt, model and response showcase

In [ ]:
subject = RUN.rows.iloc[0]
print(RUN.framing.instructions)
print()
print(RUN.prompt(subject))

In [ ]:
# One call.
CLIENT.call_llm(SPEC.model, RUN.prompt(subject), RUN.response_model, SPEC.temperature, RUN.framing.instructions)

## Run

In [ ]:
# Every call of the spec. Costs money unless the client is Fake.
written = execute(RUN, STORE, CLIENT)
written

## Check the saved results

In [ ]:
results = STORE.load(RUN.stem, RUN.response_model)
bad_turnout, bad_parties, duplicates = validate.evaluate_voting_results(results, tol=0.01)
print(f"{len(results)} results; turnout not summing to 1: {len(bad_turnout)}; "
      f"options not summing to 1: {len(bad_parties)}; duplicate options: {len(duplicates)}")
STORE.frame(RUN.stem).head()